# HDT 3: Inferencia, Naive Bayes y Regresión

**Ciencia de Datos, Sección A** · Asignada: jueves 20 de agosto · **Entrega: jueves 27 de agosto, 23:59**

**Nombre:** _(escribir aquí)_

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la mini-bitácora del final.

## Setup

Dependencias: `pip install numpy pandas matplotlib seaborn scikit-learn`. Todo es determinista (semillas fijas y datasets estáticos): las verificaciones son exactas.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

AZUL, ROJO, GRIS, LINEA = "#3A6EA5", "#B04A2E", "#75808E", "#E0E4EA"

def eje_limpio(ax):
    ax.grid(color=LINEA, lw=0.6, alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

## Parte A · Inferencia y A/B testing (0.5 pt)

El experimento del botón, en grande: 50,000 visitantes por grupo. La tasa real del control es 10.0% y la de la variante 10.5%: un efecto **pequeño** con una muestra **enorme**. Ese contraste es el tema de la parte D.

### Ejercicio 1 (0.15): la diferencia y su intervalo

(a) La diferencia observada `obs` entre variante y control. (b) El error estándar de la diferencia: $\sqrt{s_a^2/n + s_b^2/n}$ con `var(ddof=1)`. (c) El intervalo del 95%: `obs` ± 1.96 × SE. ¿El intervalo incluye el cero?

In [ ]:
rng = np.random.default_rng(9)
n_ab = 50_000
control = rng.binomial(1, 0.100, n_ab)
variante = rng.binomial(1, 0.105, n_ab)

obs = ...   # ¿Qué va aquí?
se = ...
# imprimir el intervalo

# Verificación (descomentar):
# assert round(obs, 5) == 0.00506
# assert round(obs - 1.96 * se, 5) == 0.00132

### Ejercicio 2 (0.20): test de permutación desde cero

Si el botón no importara, las etiquetas serían intercambiables. Completar la función: concatenar, revolver `reps` veces, partir en dos grupos del tamaño original, guardar la diferencia, y devolver la fracción de diferencias tan extremas como la observada.

In [ ]:
def test_permutacion(a, b, reps=3000, seed=1):
    rng_p = np.random.default_rng(seed)
    obs_ = b.mean() - a.mean()
    # ¿Qué va aquí?
    # 1) todos = concatenar a y b
    # 2) reps veces: rng_p.shuffle(todos) y partir en dos
    # 3) guardar la diferencia de medias de cada partición
    # 4) devolver la fracción con |dif| >= |obs_|
    pass

p_val = ...   # aplicarla a control y variante

# Verificación (descomentar):
# assert round(p_val, 3) == 0.008

### Ejercicio 3 (0.15): 100 tests bajo $H_0$

Ningún efecto existe: los dos grupos vienen de la misma Normal. Contar cuántos "descubrimientos" aparecen con $\alpha = 0.05$ (umbral 1.96) y cuántos sobreviven Bonferroni ($\alpha/100$, umbral 3.48).

In [ ]:
def z(a_, b_):
    s = np.sqrt(a_.var()/len(a_) + b_.var()/len(b_))
    return (b_.mean() - a_.mean()) / s

rng3 = np.random.default_rng(2)
significativos = 0
significativos_bonf = 0

for i in range(100):
    g1 = rng3.normal(0, 1, 500)
    g2 = rng3.normal(0, 1, 500)   # exactamente la misma distribución
    # ¿Qué va aquí? contar |z| > 1.96 y |z| > 3.48
    pass

# Verificación (descomentar):
# assert significativos == 3 and significativos_bonf == 0

## Parte B · Naive Bayes (0.5 pt)

Un filtro de spam para mensajes de texto, desde cero, como en la sesión 7. El corpus:

In [ ]:
textos = [
    "gana un premio reclamalo hoy mismo",
    "oferta exclusiva solo por hoy haz clic",
    "felicidades fuiste seleccionado para un premio",
    "crédito preaprobado sin requisitos aplica ya",
    "última oportunidad descuento del 90 por ciento",
    "tu cuenta será suspendida verifica tus datos",
    "recarga doble gratis si respondes ya",
    "has ganado un viaje reclama tu premio aquí",
    "nos vemos a las 8 en la u",
    "ya voy en camino llego en 10",
    "mamá dice que compres tortillas",
    "el partido quedó 2 a 1 increíble",
    "te mando el resumen de la clase en la noche",
    "feliz cumple nos vemos el sábado",
    "se me olvidó la usb me la prestas",
    "la tarea es para el martes no para hoy",
]

etiquetas = ["spam"] * 8 + ["normal"] * 8

test_x = [
    "reclama tu premio gratis haz clic ya",
    "llego en 5 nos vemos en la entrada",
    "oferta solo hoy tu crédito ya está aprobado",
    "la clase se movió para el jueves",
]
test_y = ["spam", "normal", "spam", "normal"]

print(len(textos), len(etiquetas), len(test_x))

### Ejercicio 4 (0.15): bolsa de palabras

(a) `vocab`: la lista **ordenada** de palabras únicas del corpus de entrenamiento. (b) `idx`: diccionario palabra → posición. (c) `conteos` (matriz clases × |V|) y `priors`, contando sobre `textos`.

In [ ]:
def tokenizar(texto):
    return texto.lower().split()

clases = sorted(set(etiquetas))   # ["normal", "spam"]

# ¿Qué va aquí?

# Verificación (descomentar):
# assert len(vocab) == 86
# assert conteos.shape == (2, 86)
# assert priors.tolist() == [8.0, 8.0]

### Ejercicio 5 (0.20): entrenar con Laplace y clasificar

(a) Suavizar: `conteos + 1`. (b) `log_veros` (normalizar por fila y tomar log) y `log_prior`. (c) `puntajes(texto)` que suma log-prior más log-verosimilitud de cada palabra **conocida**, y `predecir(texto)` con el argmax. (d) Clasificar los 4 mensajes de prueba.

In [ ]:
# ¿Qué va aquí?

# def puntajes(texto):
#     ...
# def predecir(texto):
#     ...

# Verificación (descomentar):
# assert [predecir(t) for t in test_x] == test_y

### Ejercicio 6 (0.15): las palabras que delatan al spam

`ratio` = log-verosimilitud de spam menos la de normal. Con `np.argsort`, extraer las 5 palabras más spam (`informativas_spam`) y las 5 más normales (`informativas_normal`).

In [ ]:
# ¿Qué va aquí?
# Pista: inv = {i: w for w, i in idx.items()}

# Verificación (descomentar):
# assert informativas_spam[0] == "premio"

## Parte C · Regresión lineal (0.8 pt)

`diamonds`: 53,940 diamantes reales. La pregunta de negocio: ¿cuánto vale un quilate?

### Ejercicio 7 (0.20): EDA mínimo

(a) `shape` y `dtypes`. (b) Media y mediana de `price`: ¿cuál es mayor y qué dice eso de la cola? (c) El histograma de `price`.

In [ ]:
diamonds = sns.load_dataset("diamonds")

# ¿Qué va aquí?

# Verificación (descomentar):
# assert diamonds.shape == (53940, 10)
# assert diamonds["price"].median() == 2401.0

### Ejercicio 8 (0.20): OLS a mano

`price ~ carat` con las ecuaciones normales: (a) `Xb` con columna de unos y `np.linalg.solve`. (b) $R^2$ a mano (`r2_precio`). (c) Comparar contra `LinearRegression`.

In [ ]:
from sklearn.linear_model import LinearRegression

x_c = diamonds["carat"].to_numpy(float)
y_p = diamonds["price"].to_numpy(float)

# ¿Qué va aquí?

# Verificación (descomentar):
# assert round(r2_precio, 3) == 0.849

### Ejercicio 9 (0.20): los residuos delatan, el log arregla

(a) Graficar residuos vs predicción del modelo del ejercicio 8. (b) Ajustar `log(price) ~ log(carat)` (`r2_log`) y graficar sus residuos. (c) Comparar los dos $R^2$ y las dos nubes.

In [ ]:
# ¿Qué va aquí?
# Pista: np.log para ambas variables; el patrón del gráfico
# de residuos está en el notebook de la sesión 8.5

# Verificación (descomentar):
# assert round(r2_log, 3) == 0.933

### Ejercicio 10 (0.20): ¿y si agregamos más features?

Regresión múltiple en la escala log: `log(price) ~ log(carat) + depth + table`. Calcular `r2_multi` y responder en un comentario: ¿cuánto mejoró respecto al ejercicio 9?

In [ ]:
# ¿Qué va aquí?

# Verificación (descomentar):
# assert round(r2_multi, 3) == 0.935

## Parte D · Criterio (0.2 pt)

### Ejercicio 11 (0.20)

Dos preguntas, 2-3 líneas cada una, **citando los números obtenidos**:

**(a)** El ejercicio 2 dio p = 0.008: el efecto del botón es estadísticamente real. El lift observado fue de medio punto porcentual (10.4% vs 9.9%). ¿Basta eso para lanzar el botón a producción? ¿Qué información falta para decidir?

**(b)** El modelo log-log del ejercicio 9 se ajustó con diamantes de hasta 5 quilates. ¿Por qué no es confiable usarlo para tasar un diamante de 8 quilates, aunque la fórmula produzca un número?

_(Responder editando esta celda)_

**Respuesta (a):** ...

**Respuesta (b):** ...

## Mini-bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".

- ...